# Evaluate Climate Models

### Parameter Settings: Change Timestep here

In [ ]:
# %% [Setup & dataset selection]

import sys
from pathlib import Path
import numpy as np
import xarray as xr

HELPER_DIR = Path("/nird/home/lbal/internship_storm_hans/helper")
if str(HELPER_DIR) not in sys.path:
    sys.path.insert(0, str(HELPER_DIR))

import config_paths as cfg
from catchment_tools import get_cached_year_range
from return_period import get_annual_maxima
from plot_style import (
    make_distribution_figure,
    make_qq_figure,
    MODEL_COLORS,
    MODEL_LABELS,
    MODEL_ORDER,
)

# Output Directory for figures
FIG_DIRS = [
    Path("/nird/home/lbal/internship_storm_hans/figures/climate_models_evaluation"),
    Path("/nird/datalake/NS9873K/lbal/figures/climate_models_evaluation"),]
for d in FIG_DIRS:
    d.mkdir(parents=True, exist_ok=True)

# Year ranges — must match what was used when producing the SMILE caches
SMILE_PERIODS = {
    "cesm2_le":          (1995, 2034),
    "gfdl_spear_med_le": (2001, 2040),}

### Data loading helper

In [ ]:
def load_annual_maxima(window_days: int) -> dict:
    """
    Load annual maxima for all models, pooled across all 5 catchments.
    Returns  {model_key: np.ndarray}.
    Prints a warning for any missing cache file (does not raise).
    """
    result = {}

    # ── ERA5 and seNorge ────────────────────────────────────────────────────
    for label, ds, res in [
        ("era5_0.5",  "era5",    "0.5x0.5"),
        ("era5_0.25", "era5",    "0.25x0.25"),
        ("senorge",   "senorge", ""),]:
        start_yr, end_yr = get_cached_year_range(ds, res, window_days)
        if start_yr is None:
            print(f"[skip] No {window_days}-day cache found for {label}")
            continue
        vals = []
        for slug in cfg.CATCHMENTS:
            nc = cfg.catchment_postproc_path(ds, res, window_days, slug, start_yr, end_yr)
            if not nc.exists():
                print(f"  [skip] {nc.name}")
                continue
            with xr.open_dataset(str(nc)) as ds_nc:
                am = get_annual_maxima(ds_nc["tp_catchment"])
                vals.extend(am.values.tolist())
        if vals:
            result[label] = np.array(vals)

    # ── SMILE ensembles ────────────────────────────────────────────────────
    for ds_key, (start_yr, end_yr) in SMILE_PERIODS.items():
        vals = []
        for slug in cfg.CATCHMENTS:
            stats_path = cfg.smile_yearmax_stats_path(
                ds_key, window_days, slug, start_yr, end_yr)
            if not stats_path.exists():
                print(f"  [skip] {stats_path.name}")
                continue
            with xr.open_dataset(str(stats_path)) as ds_s:
                am = ds_s["annual_max_yearly"].values
                vals.extend(am[np.isfinite(am)].tolist())
        if vals:
            result[ds_key] = np.array(vals)

    print(f"\nLoaded {window_days}-day data for: {list(result.keys())}")
    for k, v in result.items():
        print(f"  {MODEL_LABELS[k]:25s}  n={len(v):5d}  "
              f"mean={np.mean(v):.1f} mm  max={np.max(v):.1f} mm")
    return result

# Load data for both 1-day and 2-day windows
print("=" * 55)
print("Loading 1-day annual maxima ...")
print("=" * 55)
am_1day = load_annual_maxima(window_days=1)

print()
print("=" * 55)
print("Loading 2-day annual maxima ...")
print("=" * 55)
am_2day = load_annual_maxima(window_days=2)

### Create Distribution plots

In [5]:
out_1day_dist = [d / "distribution_1day_all_catchments.pdf" for d in FIG_DIRS]
make_distribution_figure(am_1day, window_days=1, out_paths=out_1day_dist)

    [fig]   Saved → /nird/home/lbal/internship_storm_hans/figures/climate_models_evaluation/distribution_1day_all_catchments.pdf
    [fig]   Saved → /nird/datalake/NS9873K/lbal/figures/climate_models_evaluation/distribution_1day_all_catchments.pdf


### Distribution plot: 2-day

In [6]:
out_2day_dist = [d / "distribution_2day_all_catchments.pdf" for d in FIG_DIRS]
make_distribution_figure(am_2day, window_days=2, out_paths=out_2day_dist)

    [fig]   Saved → /nird/home/lbal/internship_storm_hans/figures/climate_models_evaluation/distribution_2day_all_catchments.pdf
    [fig]   Saved → /nird/datalake/NS9873K/lbal/figures/climate_models_evaluation/distribution_2day_all_catchments.pdf


### Create Q-Q mapping Plots

In [7]:
# 1-day
reanalysis = {k: v for k, v in am_1day.items() if k != "cesm2_le"}
out = [d / "qq_mapping_cesm2_le_1day.pdf" for d in FIG_DIRS]
make_qq_figure("cesm2_le", am_1day["cesm2_le"], reanalysis, window_days=1, out_paths=out)

# 2-day
reanalysis = {k: v for k, v in am_2day.items() if k != "cesm2_le"}
out = [d / "qq_mapping_cesm2_le_2day.pdf" for d in FIG_DIRS]
make_qq_figure("cesm2_le", am_2day["cesm2_le"], reanalysis, window_days=2, out_paths=out)

    [fig]   Saved → /nird/home/lbal/internship_storm_hans/figures/climate_models_evaluation/qq_mapping_cesm2_le_1day.pdf
    [fig]   Saved → /nird/datalake/NS9873K/lbal/figures/climate_models_evaluation/qq_mapping_cesm2_le_1day.pdf
    [fig]   Saved → /nird/home/lbal/internship_storm_hans/figures/climate_models_evaluation/qq_mapping_cesm2_le_2day.pdf
    [fig]   Saved → /nird/datalake/NS9873K/lbal/figures/climate_models_evaluation/qq_mapping_cesm2_le_2day.pdf


### Q-Q mapping: GFDL-SPEAR

In [8]:
# 1-day
reanalysis = {k: v for k, v in am_1day.items() if k != "gfdl_spear_med_le"}
out = [d / "qq_mapping_gfdl_spear_med_le_1day.pdf" for d in FIG_DIRS]
make_qq_figure("gfdl_spear_med_le", am_1day["gfdl_spear_med_le"], reanalysis,
               window_days=1, out_paths=out)

# 2-day
reanalysis = {k: v for k, v in am_2day.items() if k != "gfdl_spear_med_le"}
out = [d / "qq_mapping_gfdl_spear_med_le_2day.pdf" for d in FIG_DIRS]
make_qq_figure("gfdl_spear_med_le", am_2day["gfdl_spear_med_le"], reanalysis,
               window_days=2, out_paths=out)

    [fig]   Saved → /nird/home/lbal/internship_storm_hans/figures/climate_models_evaluation/qq_mapping_gfdl_spear_med_le_1day.pdf
    [fig]   Saved → /nird/datalake/NS9873K/lbal/figures/climate_models_evaluation/qq_mapping_gfdl_spear_med_le_1day.pdf
    [fig]   Saved → /nird/home/lbal/internship_storm_hans/figures/climate_models_evaluation/qq_mapping_gfdl_spear_med_le_2day.pdf
    [fig]   Saved → /nird/datalake/NS9873K/lbal/figures/climate_models_evaluation/qq_mapping_gfdl_spear_med_le_2day.pdf
